<a href="https://colab.research.google.com/github/dilbal/db125msc26project/blob/main/Tang2024_Replication.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tang (2024): VARLiNGAM trading replication

An exploratory replication comparing a VARLiNGAM-selected predictor set with an own-price-only baseline. Both use expanding-window linear regression to predict prices and rank stocks for a daily long–short strategy.

This notebook uses a manually specified stock subset and implements VARLiNGAM only. It should be read as a partial replication; exact agreement with the reference implementation has not been established here.

## Notebook guide

1. Install dependencies and import libraries.
2. Set the sample, lag and cost assumptions.
3. Download, fill and split price data.
4. Estimate the VARLiNGAM graph and select predictors.
5. Generate expanding-window price forecasts.
6. Backtest the long–short portfolios and download SPY.
7. Plot the parameter sweep and inspect its summary.


## 1. Dependencies and imports

Install `yfinance`, `lingam` and `networkx`; import the numerical, regression and plotting libraries used below. Package versions are not pinned, and warnings are suppressed globally, so results and diagnostics may vary across environments.


In [ ]:
!pip install yfinance lingam networkx -q

import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import networkx as nx
import warnings
import time
warnings.filterwarnings('ignore')


## 2. Experiment settings

- Request daily data from September 2009 to December 2019; the download end date is exclusive.
- Reserve the first 80% of price rows for initial training.
- Use `LAG = 1` for regression features and pass it to VARLiNGAM.
- Set the default daily cost deduction to `0.001` (0.10%); later compare three cost levels.
- Use the candidate tickers below. The retained stock count is determined after downloading and filling data.

The ticker list is manually selected, rather than reconstructed from historical index membership.


In [ ]:
# ============================================================
# Experiment settings
# ============================================================
START_DATE = '2009-09-01'
END_DATE   = '2019-12-31'
TRAIN_FRAC = 0.80          # 80/20 split
LAG        = 1             # regression lag count; also passed to VARLiNGAM
TC         = 0.001         # fixed 0.10% deduction per retained trading day
SEED       = 42

# Candidate universe; availability and retained count are checked after download
SP500_TICKERS = [
    'AAPL','MSFT','GOOGL','INTC','IBM','ORCL','QCOM','TXN','ADBE','CRM',
    'JPM','BAC','WFC','C','GS','MS','AXP','BLK','USB','COF',
    'AIG','MET','PRU','ALL','TRV',
    'JNJ','UNH','PFE','ABT','MRK','AMGN','GILD','BMY','MDT','ABBV',
    'AMZN','HD','MCD','NKE','SBUX','TGT','LOW','F','GM',
    'PG','KO','PEP','WMT','COST','MO','PM','CL','MDLZ',
    'GE','BA','HON','MMM','CAT','UNP','UPS','EMR','DE','GD','LMT','RTX',
    'XOM','CVX','COP','SLB','OXY','HAL','PSX','MPC','VLO',
    'DUK','SO','D','EXC','AEP','NEE','XEL',
    'AMT','SPG','PSA',
    'T','VZ',
    'LIN','DD','NEM',
]

## 3. Price data and chronological split

Download adjusted daily closing prices, interpolate missing values across the full sample, drop columns with remaining missing values, then split chronologically.


In [ ]:
# ============================================================
# Download, fill missing prices and split chronologically
# ============================================================
print('Downloading data...')
raw    = yf.download(SP500_TICKERS, start=START_DATE, end=END_DATE,
                     interval='1d', auto_adjust=True, progress=False)
prices = raw['Close'] if isinstance(raw.columns, pd.MultiIndex) else raw[['Close']]

# Fill across the full sample without a gap limit, then drop incomplete columns.
# Both-direction filling can use future prices and extend endpoints.
prices = prices.interpolate(method='linear', limit_direction='both').dropna(axis=1)

N, T   = prices.shape[1], prices.shape[0]
stocks = prices.columns.tolist()

split_idx    = int(T * TRAIN_FRAC)
train_prices = prices.iloc[:split_idx]
test_prices  = prices.iloc[split_idx:]

print(f'After cleaning: {N} stocks, {T} days')
print(f'Training: {len(train_prices)} days ({train_prices.index[0].date()} to {train_prices.index[-1].date()})')
print(f'Test:     {len(test_prices)} days  ({test_prices.index[0].date()} to {test_prices.index[-1].date()})')

## 4. VARLiNGAM graph and predictor sets

* Fit VARLiNGAM to training-period price levels.
* Sum the absolute fitted adjacency matrices, including contemporaneous and lagged effects, then transpose the summary matrix to create directed predictor-to-target edges.
* Removing edge weights discards coefficient magnitude; taking absolute values and summing also discards sign and lag identity.

* For each stock, use its external graph predecessors as predictors.
* If none exist, use its own price. The control always uses the stock’s own price. These predictor sets remain fixed during the test period.

* Edges are model estimates under VARLiNGAM assumptions, rather than independently established causal relationships.
* The printed parent-count statistics include the own-price fallback.


In [ ]:
# Fit the training graph, then extract fixed predictor sets.
import lingam

# Fit on training-period price levels.
train_prices_np = train_prices.values

print(f'Fitting VARLiNGAM(lags={LAG}) on {len(train_prices_np)} x {N} data...')
print('(Runtime depends on the data and environment.)')
t0        = time.time()
var_model = lingam.VARLiNGAM(lags=LAG, random_state=SEED)
var_model.fit(train_prices_np)
print(f'Done in {time.time()-t0:.0f}s')

# Sum absolute coefficients across contemporaneous (k=0) and lagged matrices.
# B_k[i,j] is the fitted effect from variable j to variable i at lag k.
summary_matrix = np.sum(np.abs(var_model.adjacency_matrices_), axis=0)  # (N, N)

# Transpose so a nonzero fitted j-to-i relationship becomes edge j→i.
causal_graph = nx.from_numpy_array(summary_matrix.T, create_using=nx.DiGraph)
for u, v, d in causal_graph.edges(data=True):
    d.pop('weight', None)  # retain connectivity only

stock_to_idx = {s: i for i, s in enumerate(stocks)}

# Build parent sets: graph predecessors excluding self-loops
parents_causal = {}
parents_self   = {}
for i, stock in enumerate(stocks):
    pa_idx = [int(u) for u in causal_graph.predecessors(i) if int(u) != i]
    parents_causal[stock] = [stocks[j] for j in pa_idx] if pa_idx else [stock]
    parents_self[stock]   = [stock]

n_pa = [len(v) for v in parents_causal.values()]
print(f'Causal parents/stock: mean={np.mean(n_pa):.1f}, median={np.median(n_pa):.1f}, max={max(n_pa)}')
print(f'Stocks with >= 1 external parent: {sum(1 for s in stocks if parents_causal[s] != [s])}/{N}')


## 5. Expanding-window forecasts

* For feature row `j`, concatenate the selected predictors’ prices from rows `j` through `j + LAG - 1`; the target is the stock price at row `j + LAG`.

* At each forecast step, fit linear regression using only earlier feature–target rows. Convert the predicted price into a return relative to the preceding price. Output entry `pred_rets[k]` forecasts the return from test day `k - 1` to day `k`.

* The initial regression cutoff is computed in the shortened target array, while output rows are mapped using the original price split. Some entries can remain missing. Forecast fitting excludes the current target, but the full-sample preprocessing caveat still applies.


In [ ]:
# Build lagged price features and refit regressions at each forecast step.

all_prices_np = np.asarray(pd.concat([train_prices, test_prices]).values, dtype=np.float64)
T_total       = all_prices_np.shape[0]
T_test        = len(test_prices)

# Initial fit length in target-array coordinates (Y = target[LAG:]).
# Map forecasts back to the original price split using t_rel below.
train_length = int((T_total - LAG) * TRAIN_FRAC)


def compute_pred_returns(parents_dict, label):
    pred_rets = np.full((T_test, N), np.nan)
    print(f'{label}: predicting {N} stocks x {T_test} test days...', flush=True)
    t0 = time.time()

    for i, stock in enumerate(stocks):
        pa      = parents_dict[stock]
        pa_idx  = [stock_to_idx[s] for s in pa]
        n_pa    = len(pa_idx)
        causes  = all_prices_np[:, pa_idx]   # (T_total, n_pa)
        target  = all_prices_np[:, i]        # (T_total,)

        # Build full feature matrix: X[j] = concat of causes[j:j+LAG] (LAG rows)
        # for j in range(T_total - LAG); Y[j] = target[j + LAG]
        T_feat = T_total - LAG
        X = np.empty((T_feat, LAG * n_pa), dtype=np.float64)
        for j in range(T_feat):
            X[j] = causes[j:j + LAG, :].ravel()
        Y = target[LAG:]  # Y[j] = price at time j + LAG

        # Expanding-window: for each t from train_length onward, fit on X[:t],Y[:t]
        for t in range(train_length, T_feat):
            lr = LinearRegression(fit_intercept=True)
            lr.fit(X[:t], Y[:t])
            pred_price = float(lr.predict(X[t].reshape(1, -1))[0])
            # t in Y-space → predicts price at original index t + LAG
            # current price (used for return) = price at t + LAG - 1
            curr_price = all_prices_np[t + LAG - 1, i]
            # test period position: original index t+LAG maps to test day t+LAG - split_idx
            t_rel = (t + LAG) - split_idx
            if 0 <= t_rel < T_test and curr_price > 0:
                pred_rets[t_rel, i] = (pred_price - curr_price) / curr_price

        if (i + 1) % 20 == 0:
            print(f'  {i+1}/{N} stocks  {time.time()-t0:.0f}s', flush=True)

    print(f'  Done in {time.time()-t0:.1f}s')
    return pred_rets


pred_causal = compute_pred_returns(parents_causal, 'Causal discovery')
pred_self   = compute_pred_returns(parents_self,   'Self-cause only')


## 6. Long–short backtest and SPY benchmark

* For each `eta`, rank the available predictions, buy the top `eta` stocks and short the bottom `eta`. Each leg is equally weighted: total long weight +1 and short weight −1, giving zero net exposure and gross exposure 2.

* The daily return is the long-leg mean minus the short-leg mean, less a **fixed daily deduction** `tc`. This is not a turnover-based or per-trade cost calculation; financing and stock-borrow costs are not separately modeled. Compound the retained daily returns and annualise using 252 days. Days with too few predictions are skipped rather than recorded as cash days.

* The sweep uses `eta = 1, …, floor(N/8)` and daily deductions of 0%, 0.05% and 0.10%. SPY provides a buy-and-hold comparison; its exact dates and annualisation convention need alignment before a strict comparison.


In [ ]:
# ============================================================
# Long–short returns with a fixed daily cost deduction
# ============================================================
prices_np_test = all_prices_np[split_idx:]


def run_strategy(pred_rets, eta_values, tc=TC):
    """Compound retained daily long-minus-short returns, less tc, for each eta."""
    results = {}
    for eta in eta_values:
        if eta > N // 2:
            results[eta] = np.nan
            continue
        daily_rets = []
        for t_rel in range(T_test - 1):
            # Entry k forecasts the return ending on test day k.
            # Use k=t_rel+1 to match the realised interval below.
            pr    = pred_rets[t_rel + 1]
            valid = ~np.isnan(pr)
            if valid.sum() < 2 * eta:
                continue
            valid_ranked = [i for i in np.argsort(pr) if valid[i]]
            winners      = valid_ranked[-eta:]
            losers       = valid_ranked[:eta]
            r_next       = ((prices_np_test[t_rel + 1] - prices_np_test[t_rel])
                            / prices_np_test[t_rel])
            daily_rets.append(float(np.mean(r_next[winners])) -
                              float(np.mean(r_next[losers])) - tc)
        if daily_rets:
            cum          = float(np.prod(1.0 + np.array(daily_rets)))
            results[eta] = cum ** (252.0 / len(daily_rets)) - 1.0
        else:
            results[eta] = np.nan
    return results


eta_values = list(range(1, N // 8 + 1))
TC_LEVELS  = [0.000, 0.0005, 0.001]

# SPY buy-and-hold benchmark; download end is exclusive.
# This exponent uses price-row count, rather than return-interval count.
spy_raw    = yf.download('SPY', start=str(test_prices.index[0].date()),
                         end=str(test_prices.index[-1].date()),
                         interval='1d', auto_adjust=True, progress=False)
spy_prices = spy_raw['Close'].squeeze()
spy_annual = float((spy_prices.iloc[-1] / spy_prices.iloc[0]) ** (252 / len(spy_prices)) - 1)
print(f'SPY annualised (requested test period): {spy_annual:.3f}')

## 7. Parameter sweep and summary

* Plot annualised returns for the two predictor sets across `eta` and the three daily cost deductions.
* The dotted line is SPY; returns are displayed as decimals (0.10 means 10%).

* The summary reports each strategy’s highest return across the tested `eta` values.
* This selects the best setting on the same test sample, so it is an exploratory maximum rather than an independently validated performance estimate.
* The SPY value is repeated unchanged across columns; strategy cost deductions are not applied to it.

* The reference headline printed below is retained from the original notebook for context. Differences in universe, dates and implementation prevent a like-for-like comparison.


In [ ]:
# ============================================================
# Plot the parameter sweep and report test-sample maxima
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
colors    = ['green', 'orange', 'red']
labels    = ['TC=0% (gross signal)', 'TC=0.05%/day', 'TC=0.10%/day']
eta_arr   = np.array(eta_values)

for ax, (pred_rets, strategy_label) in zip(
        axes, [(pred_causal, 'Causal Discovery (VARLiNGAM)'),
               (pred_self,   'Self-cause only')]):
    for tc, label, color in zip(TC_LEVELS, labels, colors):
        res     = run_strategy(pred_rets, eta_values, tc)
        ret_arr = np.array([res.get(e, np.nan) for e in eta_values])
        ax.plot(eta_arr, ret_arr, lw=2.5, color=color, label=label)
    ax.axhline(spy_annual, color='blue', ls='dotted', lw=2,
               label=f'SPY ({spy_annual:.2f})')
    ax.axhline(0, color='black', lw=0.5)
    ax.set_xlabel('Number of winners/losers (η)', fontsize=12)
    ax.set_ylabel('Annualised return', fontsize=12)
    ax.set_title(f'{strategy_label}\n{N} S&P 500 stocks | lag={LAG}',
                 fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Tang (2024) Replication — VARLiNGAM Long-Short Strategy\n'
             f'Test period: {test_prices.index[0].date()} to {test_prices.index[-1].date()}',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# Exploratory maximum over eta on the same test sample
print('\nSUMMARY — Best annualised return across η')
print(f'{"Strategy":<28} {"TC=0":>8} {"TC=0.05%":>10} {"TC=0.10%":>10}')
print('-' * 60)
for pred_rets, lbl in [(pred_causal, 'Causal (VARLiNGAM)'), (pred_self, 'Self-cause')]:
    row = []
    for tc in TC_LEVELS:
        res = run_strategy(pred_rets, eta_values, tc)
        row.append(float(np.nanmax([v for v in res.values() if v == v])))
    print(f'{lbl:<28} {row[0]:>8.3f} {row[1]:>10.3f} {row[2]:>10.3f}')
print(f'{"SPY buy-and-hold":<28} {spy_annual:>8.3f} {spy_annual:>10.3f} {spy_annual:>10.3f}')
print(f'\nOriginal reference note (446 stocks, lag=1):    — best ~2.5 at η≈10-15 (TC=0.1%)')